# **Phase 1: Faster R-CNN Training on DAWN (In-domain DAWN -> DAWN)**

This notebook trains and evaluates a Faster R-CNN baseline on the DAWN dataset under an in-domain setting.

Notes:
- This preliminary implementation uses TorchVision's official Faster R-CNN implementation.
- Detectron2 is planned for the final extended experiments.

Experiment summary:
- Model: Faster R-CNN (TorchVision implementation)
- Protocol: In-domain (DAWN → DAWN)
- Dataset: DAWN
- Format: Pascal VOC
- Subset setting: reduced train/val/weather subsets
- Purpose: preliminary two-stage baseline for comparison with YOLOv11

### Connect to Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **Paths**

In [2]:
from pathlib import Path

RANDOM_SEED = 456

PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")

DAWN_VOC_ROOT = PROJECT_ROOT / "Datasets" / "processed" / "dawn_voc"
RUNS_ROOT = PROJECT_ROOT / "Runs" / "faster_rcnn"
EXPERIMENT_NAME = "dawn_indomain_fasterrcnn_r50fpn_seed456"
OUTPUT_DIR = RUNS_ROOT / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT, PROJECT_ROOT.exists())
print("DAWN_VOC_ROOT:", DAWN_VOC_ROOT, DAWN_VOC_ROOT.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

print("JPEGImages:", (DAWN_VOC_ROOT / "JPEGImages").exists())
print("Annotations:", (DAWN_VOC_ROOT / "Annotations").exists())
print("ImageSets/Main:", (DAWN_VOC_ROOT / "ImageSets" / "Main").exists())

for split in ["train", "val", "test_fog", "test_rain", "test_snow"]:
    split_file = DAWN_VOC_ROOT / "ImageSets" / "Main" / f"{split}.txt"
    print(split, split_file.exists(), split_file)

PROJECT_ROOT: /content/drive/MyDrive/Dissertation True
DAWN_VOC_ROOT: /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_voc True
OUTPUT_DIR: /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed456
JPEGImages: True
Annotations: True
ImageSets/Main: True
train True /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main/train.txt
val True /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main/val.txt
test_fog True /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main/test_fog.txt
test_rain True /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main/test_rain.txt
test_snow True /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main/test_snow.txt


# **Install Detectron**

In [3]:
# Fix numpy compatibility
!pip install -q --force-reinstall numpy==1.26.4

# Build dependencies
!pip install -q setuptools==68.0.0 wheel cython

#Import detectron from the source
%cd /content

import os

if not os.path.exists("/content/detectron2"):
    !git clone https://github.com/facebookresearch/detectron2.git

%cd /content/detectron2
!python -m pip install --no-build-isolation -e .
%cd /content

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires num

In [ ]:
import os
os.kill(os.getpid(), 9)

In [4]:
import numpy as np
print(np.__version__)

1.26.4


# **Imports libraires**

In [5]:
import detectron2
print("Detectron2 imported successfully.")

Detectron2 imported successfully.


### Force detectron path

In [6]:
import sys

# Remove all cached detectron2 modules
for module_name in list(sys.modules.keys()):
    if module_name.startswith("detectron2"):
        del sys.modules[module_name]

# Force GitHub Detectron2 path
sys.path = [p for p in sys.path if "detectron2" not in p.lower()]
sys.path.insert(0, "/content/detectron2")

import detectron2
print(detectron2.__file__)

/content/detectron2/detectron2/__init__.py


In [7]:
!pip install -q torchmetrics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.3 MB/s eta 0:00:00


In [8]:
import os
import json
import random
import math
from pathlib import Path
import torch
import time

import numpy as np
import pandas as pd
import torch
import cv2

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor, hooks
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_pascal_voc
from detectron2.utils.logger import setup_logger
from detectron2.data.datasets import register_pascal_voc
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.structures import BoxMode
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

setup_logger()

<Logger detectron2 (DEBUG)>

# **Registration + Verification**

In [13]:
CLASS_NAMES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]

datasets = {
    "dawn_train": "train",
    "dawn_val": "val",
    "dawn_test_fog": "test_fog",
    "dawn_test_rain": "test_rain",
    "dawn_test_snow": "test_snow",
}

def reset_detectron2_dataset(name):
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
    if name in MetadataCatalog.list():
        MetadataCatalog.remove(name)

for dataset_name in datasets:
    reset_detectron2_dataset(dataset_name)

for dataset_name, split_name in datasets.items():
    register_pascal_voc(
        name=dataset_name,
        dirname=str(DAWN_VOC_ROOT),
        split=split_name,
        year="",
        class_names=CLASS_NAMES
    )

print("DAWN datasets registered.")

DAWN datasets registered.


In [14]:
for dataset_name in datasets:
    d = DatasetCatalog.get(dataset_name)
    print(dataset_name, ":", len(d), "images")
    print(MetadataCatalog.get(dataset_name).thing_classes)

dawn_train : 423 images
['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
dawn_val : 140 images
['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
dawn_test_fog : 60 images
['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
dawn_test_rain : 40 images
['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
dawn_test_snow : 40 images
['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


# **Training Configuration**

In [15]:
seed = RANDOM_SEED

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

IMS_PER_BATCH = 4
EPOCHS = 100

train_size = len(DatasetCatalog.get("dawn_train"))
iters_per_epoch = math.ceil(train_size / IMS_PER_BATCH)
MAX_ITER = EPOCHS * iters_per_epoch

CHECKPOINT_PERIOD = iters_per_epoch * 10
EVAL_PERIOD = iters_per_epoch

print("Train size:", train_size)
print("Iterations per epoch:", iters_per_epoch)
print("Max iterations:", MAX_ITER)
print("Checkpoint period:", CHECKPOINT_PERIOD)
print("Eval period:", EVAL_PERIOD)

Train size: 423
Iterations per epoch: 106
Max iterations: 10600
Checkpoint period: 1060
Eval period: 106


# **Trainer with BestCheckPointer**

In [17]:
class TrainerWithBestCheckpoint(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "eval", dataset_name)
        return COCOEvaluator(dataset_name, cfg, False, output_folder)

    def build_hooks(self):
        hook_list = super().build_hooks()

        hook_list.insert(
            -1,
            hooks.BestCheckpointer(
                self.cfg.TEST.EVAL_PERIOD,
                DetectionCheckpointer(self.model, self.cfg.OUTPUT_DIR),
                "bbox/AP",
                mode="max",
                file_prefix="model_best"
            )
        )

        return hook_list

### **Build cfg**

In [16]:
cfg = get_cfg()

cfg.merge_from_file(
    model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")
)

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
)

cfg.DATASETS.TRAIN = ("dawn_train",)
cfg.DATASETS.TEST = ("dawn_val",)

cfg.DATALOADER.NUM_WORKERS = 2

cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CLASS_NAMES)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

cfg.SOLVER.IMS_PER_BATCH = IMS_PER_BATCH
cfg.SOLVER.BASE_LR = 0.001
cfg.SOLVER.MOMENTUM = 0.9
cfg.SOLVER.WEIGHT_DECAY = 0.0001

cfg.SOLVER.MAX_ITER = MAX_ITER
cfg.SOLVER.STEPS = []
cfg.SOLVER.CHECKPOINT_PERIOD = CHECKPOINT_PERIOD

cfg.TEST.EVAL_PERIOD = EVAL_PERIOD

cfg.OUTPUT_DIR = str(OUTPUT_DIR)
cfg.SEED = seed

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

with open(OUTPUT_DIR / "config.yaml", "w") as f:
    f.write(cfg.dump())

print("Config ready.")
print("MAX_ITER:", cfg.SOLVER.MAX_ITER)
print("OUTPUT_DIR:", cfg.OUTPUT_DIR)

Config ready.
MAX_ITER: 10600
OUTPUT_DIR: /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed456


### **Train**

In [ ]:
trainer = TrainerWithBestCheckpoint(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

[07/16 06:29:16 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

model_final_280758.pkl: 167MB [00:00, 327MB/s]                           
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}


[07/16 06:29:17 d2.engine.train_loop]: Starting training from iteration 0


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0716 06:29:22.068000 3324 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


Streaming output truncated to the last 5000 lines.
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.067
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.091
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.041
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.081
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.146
[07/16 06:34:36 d2.evaluation.coco_evaluation]: Evaluation results for bbox: 
|  AP   |  AP50  |  AP75  |  APs  |  APm  |  APl   |
|:-----:|:------:|:------:|:-----:|:-----:|:------:|
| 8.159 | 12.316 | 8.998  | 3.327 | 7.140 | 13.404 |
[07/16 06:34:36 d2.evaluation.coco_evaluation]: Per-category bbox AP: 
| category   | AP    | category   | AP    | category   | AP     |
|:-----------|:------|:-----------|:------|:-----------|:-------|
| person     | 0.000 | bicycle    | 0.000 | car        | 48.954 |
| motorcycle | 0.000 | bus        | 0.000 |

### **Load best model**

In [18]:
BEST_MODEL = OUTPUT_DIR / "model_best.pth"

print("Best model exists:", BEST_MODEL.exists())

cfg.MODEL.WEIGHTS = str(BEST_MODEL)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

predictor = DefaultPredictor(cfg)

Best model exists: True
[07/16 09:54:23 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed456/model_best.pth ...


# **Evaluation Global and weather-specific**

In [19]:
eval_datasets = {
    "fog": "dawn_test_fog",
    "rain": "dawn_test_rain",
    "snow": "dawn_test_snow",
}

results_rows = []

for condition, dataset_name in eval_datasets.items():
    print(f"\nEvaluating DAWN {condition.upper()}...")

    evaluator = COCOEvaluator(
        dataset_name,
        cfg,
        False,
        output_dir=str(OUTPUT_DIR / "evaluation" / condition)
    )

    loader = build_detection_test_loader(cfg, dataset_name)
    eval_results = inference_on_dataset(predictor.model, loader, evaluator)

    bbox = eval_results["bbox"]

    results_rows.append({
        "dataset": "DAWN",
        "model": "Faster R-CNN",
        "experiment": "in_domain",
        "seed": RANDOM_SEED,
        "condition": condition,
        "mAP50-95": bbox["AP"] / 100,
        "mAP50": bbox["AP50"] / 100,
        "mAP75": bbox["AP75"] / 100,
    })

results_df = pd.DataFrame(results_rows)

global_row = {
    "dataset": "DAWN",
    "model": "Faster R-CNN",
    "experiment": "in_domain",
    "seed": RANDOM_SEED,
    "condition": "global",
    "mAP50-95": results_df["mAP50-95"].mean(),
    "mAP50": results_df["mAP50"].mean(),
    "mAP75": results_df["mAP75"].mean(),
}

results_df = pd.concat(
    [pd.DataFrame([global_row]), results_df],
    ignore_index=True
)

results_df


Evaluating DAWN FOG...
WARNING [07/16 09:54:35 d2.evaluation.coco_evaluation]: COCO Evaluator instantiated using config, this is deprecated behavior. Please pass in explicit arguments instead.
[07/16 09:54:35 d2.evaluation.coco_evaluation]: Trying to convert 'dawn_test_fog' to COCO format ...
WARNING [07/16 09:54:37 d2.data.datasets.coco]: Using previously cached COCO format annotations at '/content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed456/evaluation/fog/dawn_test_fog_coco_format.json'. You need to clear the cache file if your dataset has been modified.
[07/16 09:54:37 d2.data.build]: Distribution of instances among all 6 categories:
|  category  | #instances   |  category  | #instances   |  category  | #instances   |
|:----------:|:-------------|:----------:|:-------------|:----------:|:-------------|
|   person   | 15           |  bicycle   | 0            |    car     | 357          |
| motorcycle | 3            |    bus     | 11           

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0716 09:54:40.087000 2884 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


[07/16 09:54:43 d2.evaluation.evaluator]: Inference done 11/60. Dataloading: 0.2485 s/iter. Inference: 0.0625 s/iter. Eval: 0.0003 s/iter. Total: 0.3112 s/iter. ETA=0:00:15
[07/16 09:54:48 d2.evaluation.evaluator]: Inference done 22/60. Dataloading: 0.3680 s/iter. Inference: 0.0469 s/iter. Eval: 0.0003 s/iter. Total: 0.4152 s/iter. ETA=0:00:15
[07/16 09:54:54 d2.evaluation.evaluator]: Inference done 36/60. Dataloading: 0.3565 s/iter. Inference: 0.0439 s/iter. Eval: 0.0003 s/iter. Total: 0.4008 s/iter. ETA=0:00:09
[07/16 09:54:59 d2.evaluation.evaluator]: Inference done 49/60. Dataloading: 0.3541 s/iter. Inference: 0.0418 s/iter. Eval: 0.0003 s/iter. Total: 0.3962 s/iter. ETA=0:00:04
[07/16 09:55:03 d2.evaluation.evaluator]: Total inference time: 0:00:22.071854 (0.401306 s / iter per device, on 1 devices)
[07/16 09:55:03 d2.evaluation.evaluator]: Total inference pure compute time: 0:00:02 (0.040463 s / iter per device, on 1 devices)
[07/16 09:55:04 d2.evaluation.coco_evaluation]: Prepar

,dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75
0,DAWN,Faster R-CNN,in_domain,456,global,0.422588,0.664261,0.461324
1,DAWN,Faster R-CNN,in_domain,456,fog,0.380351,0.622891,0.358460
2,DAWN,Faster R-CNN,in_domain,456,rain,0.390254,0.620438,0.471364
3,DAWN,Faster R-CNN,in_domain,456,snow,0.497159,0.749454,0.554149


### **FPS**

In [20]:
def get_image_paths_from_split(voc_root, split_name):
    split_file = voc_root / "ImageSets" / "Main" / f"{split_name}.txt"

    with open(split_file, "r") as f:
        image_ids = [line.strip() for line in f if line.strip()]

    return [
        voc_root / "JPEGImages" / f"{image_id}.jpg"
        for image_id in image_ids
        if (voc_root / "JPEGImages" / f"{image_id}.jpg").exists()
    ]


def measure_fps(predictor, image_paths, warmup=20):
    images = []

    for p in image_paths:
        img = cv2.imread(str(p))
        if img is not None:
            images.append(img)

    for img in images[:warmup]:
        _ = predictor(img)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []

    for img in images:
        start = time.perf_counter()
        _ = predictor(img)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()
        times.append(end - start)

    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms

    return avg_ms, fps

In [21]:
split_mapping = {
    "fog": "test_fog",
    "rain": "test_rain",
    "snow": "test_snow",
}

fps_data = {}

for condition, split_name in split_mapping.items():
    image_paths = get_image_paths_from_split(DAWN_VOC_ROOT, split_name)

    inference_ms, fps = measure_fps(
        predictor,
        image_paths,
        warmup=20
    )

    fps_data[condition] = {
        "Inference_ms_per_image": inference_ms,
        "FPS": fps
    }

    print(condition, inference_ms, fps)

fog 49.55869786666275 20.178092707166993
rain 49.95005257500793 20.019998947915848
snow 49.21201652499576 20.320240270830602


In [22]:
for idx, row in results_df.iterrows():
    condition = row["condition"]

    if condition == "global":
        results_df.loc[idx, "Inference_ms_per_image"] = np.mean(
            [v["Inference_ms_per_image"] for v in fps_data.values()]
        )
        results_df.loc[idx, "FPS"] = np.mean(
            [v["FPS"] for v in fps_data.values()]
        )
    else:
        results_df.loc[idx, "Inference_ms_per_image"] = fps_data[condition]["Inference_ms_per_image"]
        results_df.loc[idx, "FPS"] = fps_data[condition]["FPS"]

results_df

,dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75,Inference_ms_per_image,FPS
0,DAWN,Faster R-CNN,in_domain,456,global,0.422588,0.664261,0.461324,49.573589,20.172777
1,DAWN,Faster R-CNN,in_domain,456,fog,0.380351,0.622891,0.358460,49.558698,20.178093
2,DAWN,Faster R-CNN,in_domain,456,rain,0.390254,0.620438,0.471364,49.950053,20.019999
3,DAWN,Faster R-CNN,in_domain,456,snow,0.497159,0.749454,0.554149,49.212017,20.320240


### **Sav**

# **Save training loss**

In [24]:
results_csv = OUTPUT_DIR / "dawn_faster_rcnn_in_domain_seed456_results_summary.csv"
results_json = OUTPUT_DIR / "dawn_faster_rcnn_in_domain_seed456_results_summary.json"

results_df.to_csv(results_csv, index=False)

with open(results_json, "w") as f:
    json.dump(
        results_df.to_dict(orient="records"),
        f,
        indent=4
    )

print("Saved:")
print(results_csv)
print(results_json)

Saved:
/content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed456/dawn_faster_rcnn_in_domain_seed456_results_summary.csv
/content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed456/dawn_faster_rcnn_in_domain_seed456_results_summary.json
